## Import 

In [1]:
import os
import gc
import math
import pickle 
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interp
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from tableone import TableOne

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_data  = "./Data/"

### Reading Data

In [3]:
df_ehr = pd.read_csv(path_data + 'all_24h_data.csv', low_memory=False, index_col=False)
df_ehr = df_ehr[df_ehr.datetime != 0.0]
df_ehr.head(2)

In [4]:
print(df_ehr.patientid.nunique())
print(df_ehr.shape)

33811
(706601, 797)


### Drop Columns

In [5]:
del_diff_columns = [col for col in list(df_ehr.columns) if '_diff' in col]
del_tslm_columns = [col for col in list(df_ehr.columns) if '_tslm' in col]

del_diff_columns.extend(del_tslm_columns)
df_ehr = df_ehr.drop(columns=del_diff_columns)

print(df_ehr.patientid.nunique())
print(df_ehr.shape)

### Read Demographics

In [10]:
df_demog = pd.read_csv(path_data + 'demographic_all.csv', low_memory=False, index_col=False)
df_demog = df_demog[['patientid', 'sex', 'height', 'weight', 'discharge_status', 'Los']]
df_demog['gender'] = 1
df_demog.loc[df_demog.sex == 'F', 'gender'] = 2
df_demog['icuLos_h'] = df_demog['Los'] * 24
df_demog = df_demog[~df_demog['discharge_status'].isnull()]
df_demog['icu_expire_flag'] = 0
df_demog.loc[df_demog.discharge_status == 'dead', 'icu_expire_flag'] = 1
df_demog = df_demog[['patientid', 'gender', 'height', 'weight', 'icuLos_h', 'icu_expire_flag']]
df_demog.head(3)

In [11]:
print(df_demog.patientid.nunique())
print(df_demog.shape)

33577
(33577, 6)


### Combine Data & Demographics

In [12]:
df_ehr = df_ehr.merge(df_demog, on='patientid', how='right')
df_ehr.head(2)

In [13]:
print(df_ehr.patientid.nunique())
print(df_ehr.shape)

33577
(701344, 410)


### Take first hours of ICU of patients with more than 24 hour LoS

In [14]:
observation_window = 24
observation_length = 24

In [15]:
def short_long_icustays(df, observation_window):
    
    short_stays = df[df.icuLos_h < observation_window].copy()
    short_stays = short_stays.groupby('patientid').head(observation_length).reset_index(drop=True)

    long_stays = df[df.icuLos_h >= observation_window].copy()
    long_stays = long_stays.groupby('patientid').head(observation_length).reset_index(drop=True)
    
    return long_stays, short_stays

In [16]:
df_long, df_short = short_long_icustays(df_ehr, observation_window)

In [17]:
print(df_long.patientid.nunique())
print(df_long.shape)

print(df_short.patientid.nunique())
print(df_short.shape)

16642
(399055, 410)
16935
(302289, 410)


In [18]:
del df_ehr
del df_short
gc.collect()

0

### Spliting Train - Validation - Test

In [20]:
df_long.sort_values(by=['patientid', 'datetime'], inplace=True)
df_long = df_long.reset_index(drop=True)
df_long.head(3)

### Variables Selection

In [21]:
selected_columns = ['patientid', 'Heart Rate', 'SpO2', 'Oxygen Saturation_SO2', 'Respiratory Rate', 
                    'Temperature Central', 'Invasive systolic arterial pressure',
                    'Invasive diastolic arterial pressure', 'Invasive mean arterial pressure', 
                    'Glucose', 'Creatine kinase', 'Base Excess', 'BUN', 
                    'Bicarbonate',  'Lactate', 'Hemoglobin', 'Carboxy Hemoglobin', 'pH', 'Bilirubin, Direct',
                    'pCO2', 'PO2', 'AST', 'ALT', 'Potassium', 'Sodium', 'Chloride', 
                    'Calcium', 'Phosphate', 'Magnesium', 'ST1 (ECG ST elevation)', 'ST2 (ECG ST elevation)',
                    'ST3 (ECG ST elevation)', 'FIO2', 'Peep', 'TV', 'Out Urine/h', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'Total GCS', 'RASS',
                    'age', 'gender', 
                    'icuLos_h', 'icu_expire_flag']

In [22]:
for col in selected_columns:
    temp_col = col + '_ind'
    
    if temp_col in list(df_long.columns):
        df_long.loc[df_long[temp_col] == 0, col] = np.nan

In [24]:
df_long = df_long[selected_columns]
df_long.head(3)

In [25]:
df_ehr = df_long.drop(['patientid', 'age', 'gender', 'icuLos_h'], axis=1)
df_demog = df_long[['patientid', 'age', 'gender', 'icuLos_h', 'icu_expire_flag']]

### Create TableOne

In [26]:
columns = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation_SO2', 'Respiratory Rate', 
            'Temperature Central', 'Invasive systolic arterial pressure',
            'Invasive diastolic arterial pressure', 'Invasive mean arterial pressure', 
            'Glucose', 'Creatine kinase', 'Base Excess', 'BUN', 
            'Bicarbonate',  'Lactate', 'Hemoglobin', 'Carboxy Hemoglobin', 'pH', 'Bilirubin, Direct',
            'pCO2', 'PO2', 'AST', 'ALT', 'Potassium', 'Sodium', 'Chloride', 
            'Calcium', 'Phosphate', 'Magnesium', 'ST1 (ECG ST elevation)', 'ST2 (ECG ST elevation)',
            'ST3 (ECG ST elevation)', 'FIO2', 'Peep', 'TV', 'Out Urine/h', 
            'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
            'Total GCS', 'RASS',
            'icu_expire_flag']

In [27]:
categorical = ['RASS']

In [28]:
IQR = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation_SO2', 'Respiratory Rate', 
        'Temperature Central', 'Invasive systolic arterial pressure',
        'Invasive diastolic arterial pressure', 'Invasive mean arterial pressure', 
        'Glucose', 'Creatine kinase', 'Base Excess', 'BUN', 
        'Bicarbonate',  'Lactate', 'Hemoglobin', 'Carboxy Hemoglobin', 'pH', 'Bilirubin, Direct',
        'pCO2', 'PO2', 'AST', 'ALT', 'Potassium', 'Sodium', 'Chloride', 
        'Calcium', 'Phosphate', 'Magnesium', 'ST1 (ECG ST elevation)', 'ST2 (ECG ST elevation)',
        'ST3 (ECG ST elevation)', 'FIO2', 'Peep', 'TV', 'Out Urine/h', 
        'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
        'Total GCS',]

In [29]:
EHR_table = TableOne(df_ehr, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [30]:
EHR_table

Grouped by icu_expire_flag                                                                          
                                                                             Missing               Overall                     0                     1 P-Value
n                                                                                                   399055                367308                 31747        
Heart Rate, median [Q1,Q3]                                                     10260      84.0 [71.0,96.0]      84.0 [71.0,96.0]     87.0 [71.0,102.0]  <0.001
SpO2, median [Q1,Q3]                                                           12977      97.0 [95.0,99.0]      97.0 [95.0,99.0]     98.0 [95.0,100.0]   0.570
Oxygen Saturation_SO2, median [Q1,Q3]                                         327611      98.0 [96.0,99.0]      98.0 [96.0,99.0]      97.7 [95.1,99.0]  <0.001
Respiratory Rate, median [Q1,Q3]                                               95823      17.0 [13.0,22.0]      17.0 [13.0,22.0]      16.0 [12.0,22.1]  <0.001
Temperature Central, median [Q1,Q3]                                           294783      37.1 [36.5,37.6]      37.1 [36.6,37.6]      37.0 [36.0,37.6]  <0.001
Invasive systolic arterial pressure, median [Q1,Q3]                            31395    113.0 [95.0,133.0]    113.0 [96.0,133.0]    104.0 [87.0,126.0]  <0.001
Invasive diastolic arterial pressure, median [Q1,Q3]                           31418      55.0 [48.0,64.0]      55.0 [48.0,64.0]      53.0 [46.0,62.0]  <0.001
Invasive mean arterial pressure, median [Q1,Q3]                                31381      74.0 [64.0,86.0]      75.0 [65.0,87.0]      70.0 [60.0,83.0]  <0.001
Glucose, median [Q1,Q3]                                                       285068         7.9 [6.4,9.8]         7.9 [6.4,9.8]        8.1 [6.5,10.1]  <0.001
Creatine kinase, median [Q1,Q3]                                               381791  394.0 [153.0,1104.2]  383.0 [150.0,1038.0]  547.0 [176.0,1627.0]  <0.001
Base Excess, median [Q1,Q3]                                                   327600       -1.2 [-4.0,1.1]       -1.0 [-3.6,1.2]      -3.5 [-7.6,-0.3]  <0.001
BUN, median [Q1,Q3]                                                           383524         3.4 [2.2,5.4]         3.3 [2.2,5.3]         4.2 [2.6,6.8]  <0.001
Bicarbonate, median [Q1,Q3]                                                   327542      23.1 [20.7,25.2]      23.3 [21.0,25.3]      21.1 [17.8,23.9]  <0.001
Lactate, median [Q1,Q3]                                                       325907         1.6 [1.0,2.7]         1.5 [1.0,2.5]         2.7 [1.5,5.2]  <0.001
Hemoglobin, median [Q1,Q3]                                                    319338    102.0 [90.0,118.0]    102.0 [90.0,118.0]    106.0 [92.0,125.0]  <0.001
Carboxy Hemoglobin, median [Q1,Q3]                                            327651         1.4 [1.0,1.7]         1.4 [1.1,1.7]         1.2 [0.9,1.6]  <0.001
pH, median [Q1,Q3]                                                            331771         7.4 [7.4,7.5]         7.4 [7.4,7.5]         7.4 [7.3,7.4]  <0.001
Bilirubin, Direct, median [Q1,Q3]                                             390845        8.8 [5.4,18.5]        8.8 [5.4,17.9]        9.5 [5.4,23.4]   0.001
pCO2, median [Q1,Q3]                                                          332692      35.3 [31.5,39.8]      35.4 [31.7,39.9]      34.6 [30.0,39.5]  <0.001
PO2, median [Q1,Q3]                                                           332202     98.4 [79.5,129.0]     98.6 [80.0,128.0]     98.0 [77.6,131.0]   0.058
AST, median [Q1,Q3]                                                           388424     57.0 [28.0,186.0]     54.0 [27.0,163.0]    100.0 [38.0,365.0]  <0.001
ALT, median [Q1,Q3]                                                           387528     35.0 [18.0,101.0]      34.0 [18.0,92.0]     54.0 [22.5,177.0]  <0.001
Potassium, median [Q1,Q3]                                                     32386

### Static Information

In [31]:
df_demog = df_demog.groupby('patientid').head(1)

In [33]:
df_demog.head()

In [34]:
columns = ['age', 'gender', 'icuLos_h', 'icu_expire_flag']

categorical = ['gender']

IQR = ['age', 'icuLos_h']

In [35]:
demog_table = TableOne(df_demog, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [36]:
demog_table

Grouped by icu_expire_flag                                                               
                                              Missing           Overall                 0                  1 P-Value
n                                                                 16642             15319               1323        
age, median [Q1,Q3]                                 0  65.0 [55.0,75.0]  65.0 [55.0,75.0]   70.0 [55.0,75.0]  <0.001
gender, n (%)            1                          0      10597 (63.7)       9774 (63.8)         823 (62.2)   0.259
                         2                                  6045 (36.3)       5545 (36.2)         500 (37.8)        
icuLos_h, median [Q1,Q3]                            0  48.0 [31.2,98.4]  48.0 [31.2,96.0]  62.4 [38.4,122.4]  <0.001

### Save Tables

In [37]:
# EHR_table.to_csv('./TableONe_EHR_ICU_Mortality.csv')
# demog_table.to_csv('./TableONe_DEMOG_ICU_Mortality.csv')